# Importing Libraries

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Configuration setup

## Setup Paths

In [2]:
BASE_DIR = Path.cwd().parent

ENV_PATH = BASE_DIR / ".env"
DATA_DIR = BASE_DIR / "data"
JOBS_ENRICHED_PATH = DATA_DIR / "jobs_enriched.csv"
COMPANIES_PATH = DATA_DIR / "companies.csv"

load_dotenv(ENV_PATH)

DATABASE_URL = os.getenv("DATABASE_URL")

print("Jobs file exists:", JOBS_ENRICHED_PATH.exists())
print("Companies file exists:", COMPANIES_PATH.exists())
print("Database URL loaded:", DATABASE_URL is not None)

Jobs file exists: True
Companies file exists: True
Database URL loaded: True


## Load Paths

In [6]:
companies_df = pd.read_csv(COMPANIES_PATH).fillna("")
jobs_df = pd.read_csv(JOBS_ENRICHED_PATH).fillna("")

print("Companies:", len(companies_df))
print("Jobs:", len(jobs_df))

jobs_df.head()

Companies: 5
Jobs: 1265


,company,title,location,job_url,description,ats_type,external_job_id,posted_date,date_found,description_raw,...,ats_match_score,score_label,matched_roles,matched_skills,project_relevance_hits,missing_keywords,seniority_flags,good_level_hits,score_reason,is_relevant
0,OpenAI,"Technical Program Manager, Compute Infrastructure",San Francisco,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...,About the Team The compute infrastructure team...,ashby,8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3,2026-03-12T16:38:15.322+00:00,2026-06-04T18:11:09.402369+00:00,,...,20,Low Match,applied ai,api,research,python; sql; machine learning; deep learning; ...,lead; manager; 5+ years,,Role match: applied ai | Skills matched: api |...,False
1,OpenAI,Research Engineer,San Francisco,https://jobs.ashbyhq.com/openai/240d459b-696d-...,"By applying to this role, you will be consider...",ashby,240d459b-696d-43eb-8497-fab3e56ecd9b,2025-04-05T00:03:20.653+00:00,2026-06-04T18:11:09.407935+00:00,,...,44,Low Match,research engineer,machine learning; deep learning,machine learning; research,python; sql; pytorch; tensorflow; scikit-learn...,,,Role match: research engineer | Skills matched...,False
2,OpenAI,Account Director - Tokyo,"Tokyo, Japan",https://jobs.ashbyhq.com/openai/18f58952-c242-...,About the team OpenAI’s mission is to build sa...,ashby,18f58952-c242-4562-8732-073a0ae8029e,2026-01-23T00:17:18.483+00:00,2026-06-04T18:11:09.410524+00:00,,...,17,Low Match,research engineer,,research,python; sql; machine learning; deep learning; ...,lead; director; 7+ years,,Role match: research engineer | Project releva...,False
3,OpenAI,"Software Engineer, RL Training Infra",San Francisco,https://jobs.ashbyhq.com/openai/13995549-e8cc-...,About the Team The Post-Training Frontiers tea...,ashby,13995549-e8cc-498f-9eaa-1869067ac35b,2026-05-23T02:00:50.464+00:00,2026-06-04T18:11:09.410524+00:00,,...,38,Low Match,software engineer,api,research,python; sql; machine learning; deep learning; ...,,,Role match: software engineer | Skills matched...,False
4,OpenAI,"Research Engineer, Retrieval & Search, Applied...",San Francisco,https://jobs.ashbyhq.com/openai/7322d344-9325-...,About the Team We bring OpenAI's technology to...,ashby,7322d344-9325-4a92-8445-0a2c4e9272f8,2024-03-20T21:33:20.763+00:00,2026-06-04T18:11:09.410524+00:00,,...,44,Low Match,research engineer,machine learning; api,machine learning; research,python; sql; deep learning; pytorch; tensorflo...,,,Role match: research engineer | Skills matched...,False


## Clean datetime columns

In [7]:
if "posted_datetime" in jobs_df.columns:
    jobs_df["posted_datetime"] = pd.to_datetime(
        jobs_df["posted_datetime"],
        errors="coerce",
        utc=True
    )

if "date_found" in jobs_df.columns:
    jobs_df["date_found"] = pd.to_datetime(
        jobs_df["date_found"],
        errors="coerce",
        utc=True
    )

## Connect to PostgreSQL

In [3]:
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone()[0])

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## End

# Helper Functions


In [8]:
def to_bool(value):
    return str(value).strip().lower() in ["true", "1", "yes", "y"]

In [10]:
def clean_value(value):
    if pd.isna(value):
        return None

    if value == "":
        return None

    return value

# Run

## Create Database Schema

In [4]:
create_tables_sql = """
CREATE TABLE IF NOT EXISTS companies (
    id SERIAL PRIMARY KEY,
    company TEXT UNIQUE NOT NULL,
    career_url TEXT,
    ats_type TEXT,
    ats_slug TEXT,
    workday_tenant TEXT,
    workday_site TEXT,
    workday_server TEXT,
    keywords TEXT,
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMPTZ DEFAULT NOW(),
    updated_at TIMESTAMPTZ DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS job_postings (
    id SERIAL PRIMARY KEY,
    company TEXT NOT NULL,
    title TEXT,
    location TEXT,
    job_url TEXT UNIQUE NOT NULL,
    description TEXT,
    description_raw TEXT,
    ats_type TEXT,
    external_job_id TEXT,
    posted_date TEXT,
    posted_datetime TIMESTAMPTZ,
    freshness_status TEXT,
    date_found TIMESTAMPTZ,
    first_seen TIMESTAMPTZ DEFAULT NOW(),
    last_seen TIMESTAMPTZ DEFAULT NOW(),
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMPTZ DEFAULT NOW(),
    updated_at TIMESTAMPTZ DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS job_scores (
    id SERIAL PRIMARY KEY,
    job_id INTEGER REFERENCES job_postings(id) ON DELETE CASCADE,
    keyword_match_count INTEGER,
    matched_keywords TEXT,
    role_score INTEGER,
    skill_score INTEGER,
    project_score INTEGER,
    experience_score INTEGER,
    freshness_score INTEGER,
    ats_match_score INTEGER,
    score_label TEXT,
    matched_roles TEXT,
    matched_skills TEXT,
    project_relevance_hits TEXT,
    missing_keywords TEXT,
    seniority_flags TEXT,
    good_level_hits TEXT,
    score_reason TEXT,
    is_relevant BOOLEAN DEFAULT FALSE,
    created_at TIMESTAMPTZ DEFAULT NOW(),
    updated_at TIMESTAMPTZ DEFAULT NOW(),
    UNIQUE(job_id)
);

CREATE TABLE IF NOT EXISTS application_status (
    id SERIAL PRIMARY KEY,
    job_id INTEGER UNIQUE REFERENCES job_postings(id) ON DELETE CASCADE,
    status TEXT DEFAULT 'found',
    resume_version TEXT,
    notes TEXT,
    applied_at TIMESTAMPTZ,
    created_at TIMESTAMPTZ DEFAULT NOW(),
    updated_at TIMESTAMPTZ DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS notification_log (
    id SERIAL PRIMARY KEY,
    job_id INTEGER REFERENCES job_postings(id) ON DELETE CASCADE,
    notification_type TEXT,
    score_threshold INTEGER,
    sent_to TEXT,
    sent_at TIMESTAMPTZ DEFAULT NOW()
);
"""

In [5]:
with engine.begin() as conn:
    conn.execute(text(create_tables_sql))

print("Tables created successfully.")

Tables created successfully.


## Insert Companies

In [9]:
with engine.begin() as conn:
    for _, row in companies_df.iterrows():
        conn.execute(
            text("""
                INSERT INTO companies (
                    company,
                    career_url,
                    ats_type,
                    ats_slug,
                    workday_tenant,
                    workday_site,
                    workday_server,
                    keywords,
                    is_active,
                    updated_at
                )
                 VALUES (
                    :company,
                    :career_url,
                    :ats_type,
                    :ats_slug,
                    :workday_tenant,
                    :workday_site,
                    :workday_server,
                    :keywords,
                    :is_active,
                    NOW()
                )
                ON CONFLICT (company)
                DO UPDATE SET
                    career_url = EXCLUDED.career_url,
                    ats_type = EXCLUDED.ats_type,
                    ats_slug = EXCLUDED.ats_slug,
                    workday_tenant = EXCLUDED.workday_tenant,
                    workday_site = EXCLUDED.workday_site,
                    workday_server = EXCLUDED.workday_server,
                    keywords = EXCLUDED.keywords,
                    is_active = EXCLUDED.is_active,
                    updated_at = NOW();
            """),
            {
                "company": row.get("company", ""),
                "career_url": row.get("career_url", ""),
                "ats_type": row.get("ats_type", ""),
                "ats_slug": row.get("ats_slug", ""),
                "workday_tenant": row.get("workday_tenant", ""),
                "workday_site": row.get("workday_site", ""),
                "workday_server": row.get("workday_server", ""),
                "keywords": row.get("keywords", ""),
                "is_active": to_bool(row.get("is_active", True)),
            }
        )

print("Companies inserted/updated.")

Companies inserted/updated.


## Upsert jobs and scores

In [11]:
inserted_or_updated = 0

with engine.begin() as conn:
    for _, row in jobs_df.iterrows():
        job_url = clean_value(row.get("job_url"))

        # Skip jobs without URLs because job_url is our unique identifier
        if not job_url:
            continue

        job_result = conn.execute(
            text("""
                INSERT INTO job_postings (
                    company,
                    title,
                    location,
                    job_url,
                    description,
                    description_raw,
                    ats_type,
                    external_job_id,
                    posted_date,
                    posted_datetime,
                    freshness_status,
                    date_found,
                    last_seen,
                    updated_at
                )
                VALUES (
                    :company,
                    :title,
                    :location,
                    :job_url,
                    :description,
                    :description_raw,
                    :ats_type,
                    :external_job_id,
                    :posted_date,
                    :posted_datetime,
                    :freshness_status,
                    :date_found,
                    NOW(),
                    NOW()
                )
                ON CONFLICT (job_url)
                DO UPDATE SET
                    company = EXCLUDED.company,
                    title = EXCLUDED.title,
                    location = EXCLUDED.location,
                    description = EXCLUDED.description,
                    description_raw = EXCLUDED.description_raw,
                    ats_type = EXCLUDED.ats_type,
                    external_job_id = EXCLUDED.external_job_id,
                    posted_date = EXCLUDED.posted_date,
                    posted_datetime = EXCLUDED.posted_datetime,
                    freshness_status = EXCLUDED.freshness_status,
                    date_found = EXCLUDED.date_found,
                    last_seen = NOW(),
                    updated_at = NOW()
                RETURNING id;
            """),
            {
                "company": clean_value(row.get("company")),
                "title": clean_value(row.get("title")),
                "location": clean_value(row.get("location")),
                "job_url": job_url,
                "description": clean_value(row.get("description")),
                "description_raw": clean_value(row.get("description_raw")),
                "ats_type": clean_value(row.get("ats_type")),
                "external_job_id": clean_value(row.get("external_job_id")),
                "posted_date": clean_value(row.get("posted_date")),
                "posted_datetime": clean_value(row.get("posted_datetime")),
                "freshness_status": clean_value(row.get("freshness_status")),
                "date_found": clean_value(row.get("date_found")),
            }
        )

        job_id = job_result.fetchone()[0]

        conn.execute(
            text("""
                INSERT INTO job_scores (
                    job_id,
                    keyword_match_count,
                    matched_keywords,
                    role_score,
                    skill_score,
                    project_score,
                    experience_score,
                    freshness_score,
                    ats_match_score,
                    score_label,
                    matched_roles,
                    matched_skills,
                    project_relevance_hits,
                    missing_keywords,
                    seniority_flags,
                    good_level_hits,
                    score_reason,
                    is_relevant,
                    updated_at
                )
                VALUES (
                    :job_id,
                    :keyword_match_count,
                    :matched_keywords,
                    :role_score,
                    :skill_score,
                    :project_score,
                    :experience_score,
                    :freshness_score,
                    :ats_match_score,
                    :score_label,
                    :matched_roles,
                    :matched_skills,
                    :project_relevance_hits,
                    :missing_keywords,
                    :seniority_flags,
                    :good_level_hits,
                    :score_reason,
                    :is_relevant,
                    NOW()
                )
                ON CONFLICT (job_id)
                DO UPDATE SET
                    keyword_match_count = EXCLUDED.keyword_match_count,
                    matched_keywords = EXCLUDED.matched_keywords,
                    role_score = EXCLUDED.role_score,
                    skill_score = EXCLUDED.skill_score,
                    project_score = EXCLUDED.project_score,
                    experience_score = EXCLUDED.experience_score,
                    freshness_score = EXCLUDED.freshness_score,
                    ats_match_score = EXCLUDED.ats_match_score,
                    score_label = EXCLUDED.score_label,
                    matched_roles = EXCLUDED.matched_roles,
                    matched_skills = EXCLUDED.matched_skills,
                    project_relevance_hits = EXCLUDED.project_relevance_hits,
                    missing_keywords = EXCLUDED.missing_keywords,
                    seniority_flags = EXCLUDED.seniority_flags,
                    good_level_hits = EXCLUDED.good_level_hits,
                    score_reason = EXCLUDED.score_reason,
                    is_relevant = EXCLUDED.is_relevant,
                    updated_at = NOW();
            """),
            {
                "job_id": job_id,
                "keyword_match_count": int(row.get("keyword_match_count", 0) or 0),
                "matched_keywords": clean_value(row.get("matched_keywords")),
                "role_score": int(row.get("role_score", 0) or 0),
                "skill_score": int(row.get("skill_score", 0) or 0),
                "project_score": int(row.get("project_score", 0) or 0),
                "experience_score": int(row.get("experience_score", 0) or 0),
                "freshness_score": int(row.get("freshness_score", 0) or 0),
                "ats_match_score": int(row.get("ats_match_score", 0) or 0),
                "score_label": clean_value(row.get("score_label")),
                "matched_roles": clean_value(row.get("matched_roles")),
                "matched_skills": clean_value(row.get("matched_skills")),
                "project_relevance_hits": clean_value(row.get("project_relevance_hits")),
                "missing_keywords": clean_value(row.get("missing_keywords")),
                "seniority_flags": clean_value(row.get("seniority_flags")),
                "good_level_hits": clean_value(row.get("good_level_hits")),
                "score_reason": clean_value(row.get("score_reason")),
                "is_relevant": to_bool(row.get("is_relevant", False)),
            }
        )

        conn.execute(
            text("""
                INSERT INTO application_status (job_id, status)
                VALUES (:job_id, 'found')
                ON CONFLICT (job_id)
                DO NOTHING;
            """),
            {"job_id": job_id}
        )

        inserted_or_updated += 1

print("Jobs inserted/updated:", inserted_or_updated)

Jobs inserted/updated: 1265


## Query Top jobs

### best matches from PostgreSQL

In [12]:
query = """
SELECT
    jp.company,
    jp.title,
    jp.location,
    jp.job_url,
    jp.freshness_status,
    js.ats_match_score,
    js.score_label,
    js.matched_skills,
    js.seniority_flags,
    js.score_reason,
    ast.status
FROM job_postings jp
JOIN job_scores js
    ON jp.id = js.job_id
JOIN application_status ast
    ON jp.id = ast.job_id
WHERE js.is_relevant = TRUE
ORDER BY js.ats_match_score DESC
LIMIT 25;
"""

top_jobs_df = pd.read_sql(query, engine)

top_jobs_df

,company,title,location,job_url,freshness_status,ats_match_score,score_label,matched_skills,seniority_flags,score_reason,status
0,Workday,Senior Software Engineer (Gen AI),"USA, CO, Boulder",https://workday.wd5.myworkdayjobs.com/job/USA-...,fresh,70,Good Match,python; machine learning; llm; large language ...,senior; lead; 5+ years,"Role match: ml engineer, data science, data en...",found


### Count Summary

In [13]:
summary_query = """
SELECT
    COUNT(*) AS total_jobs,
    COUNT(*) FILTER (WHERE js.is_relevant = TRUE) AS relevant_jobs,
    COUNT(*) FILTER (WHERE js.ats_match_score >= 80) AS strong_matches,
    COUNT(*) FILTER (WHERE jp.freshness_status = 'fresh') AS fresh_jobs,
    COUNT(*) FILTER (WHERE jp.freshness_status = 'unknown') AS unknown_date_jobs
FROM job_postings jp
JOIN job_scores js
    ON jp.id = js.job_id;
"""

pd.read_sql(summary_query, engine)

,total_jobs,relevant_jobs,strong_matches,fresh_jobs,unknown_date_jobs
0,1265,1,0,29,0


## End

# Save